# Testing projection TTI on GPU

In [4]:
import sys; sys.path.insert(0,"../..") # Allow import of agd from parent directory (useless if conda package installed)
#from Miscellaneous import TocTools; print(TocTools.displayTOC('SeismicNorm','Algo'))

In [5]:
from agd.Metrics.Seismic import Hooke,Thomsen
from agd.Metrics import Riemann
from agd import Sphere
from agd import LinearParallel as lp
from agd import AutomaticDifferentiation as ad
from agd.Plotting import SetTitle3D

In [6]:
import numpy as np
import copy
from matplotlib import pyplot as plt

# Start

In [40]:
x = np.array([0.1,0.2,0.3])
x=ad.Dense2.identity(constant=x)
print(f"{x=}")
r,q = Sphere.rotation3_from_ball3(x);
print(f"{r=}, {q=}")
norm = Hooke.mica[0].rotate_by(0.2,(1,2,3))
print(f"{np.round(norm.hooke,2)}")
h = norm.rotate(r).hooke
print(f"{np.round(h,2)}")

x=denseAD2(array([0.1, 0.2, 0.3]),
array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]),
array([[[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]]))
r=denseAD2(array([[ 0.19975377, -0.67097568,  0.71406587],
       [ 0.91720529,  0.38442598,  0.10464758],
       [-0.34472145,  0.63404124,  0.69221299]]),
array([[[ 0.28078815, -1.90071979, -2.85107968],
        [ 1.65125032,  1.45577857, -1.38666148],
        [ 1.47305784,  1.89963984, -0.50541867]],

       [[ 0.72464942, -0.39742323,  1.12747242],
        [-1.01515716,  0.43198177, -3.04547148],
        [-2.62212935,  1.89639997,  1.3056649 ]],

       [[ 2.09079177, -2.1588289 ,  1.34778312],
        [ 2.36294028,  1.27866604,  0.379064  ],
        [-1.1231526 , -2.24630521,  0.32398633]]]),
array([[[[  2.66009827,   0.56839707,   0.8525956 ],
         [  0.56839707,  -6.63887773,

TypeError: no implementation found for 'numpy.round_' on types that implement __array_function__: [<class 'agd.AutomaticDifferentiation.Dense2.denseAD2'>]

In [ ]:
def proj_hexagonal(c):
    """Hexagonal Hooke tensors are tetragonal, with the additional property that c66=(c11-c12)/2"""
    c11,c12,c13,c22,c23,c33,c44,c55,c66 = ( # Seismologists start at 1 ...
        c[0,0],c[0,1],c[0,2],c[1,1],c[1,2],c[2,2],c[3,3],c[4,4],c[5,5])
    α=(3*(c11+c22)+2*c12+4*c66)/8
    β=(c11+c22+6*c12-4*c66)/8
    γ=(c13+c23)/2
    δ=(c44+c55)/2
    print(α,β,γ,δ)
    return Hooke.from_orthorombic(α,β,γ,α,γ,c33,δ,δ,(α-β)/2).hooke

In [43]:
res = ((h-proj_hexagonal(h))**2).sum()/2

denseAD2(114.49169889060758,[ -83.10473533 -212.13490747   38.10342344],
[[-530.40126094  933.33339698    2.64444718]
 [ 933.33339698 1432.36189008 -158.37462179]
 [   2.64444718 -158.37462179  125.18446827]]) denseAD2(32.576723771168524,[ -5.00893898 -12.78592372   2.29659263],
[[ 15.99642884 178.69115388 -21.832552  ]
 [178.69115388 398.8667426  -65.68274229]
 [-21.832552   -65.68274229  17.62847265]]) denseAD2(41.4434064369821,[ -4.66399426 -11.90541053   2.13843588],
[[ -273.83499343  -570.63240071   112.05324543]
 [ -570.63240071 -1509.92889437   276.76236741]
 [  112.05324543   276.76236741   -44.28264673]]) denseAD2(47.65302454270658,[12.35548534 31.53887355 -5.6649755 ],
[[ -129.23735657  -669.94738454    95.01771963]
 [ -669.94738454 -1568.86930994   267.09396614]
 [   95.01771963   267.09396614   -62.3574093 ]])


In [45]:
x.value-lp.solve_AV(res.hessian(),res.gradient())

array([ 0.12001388,  0.27063976, -0.12175385])

In [33]:
np.round(h-proj_hexagonal(h),2)

114.49169889060758 32.576723771168524 41.4434064369821 47.65302454270658


array([[-45.58,  -4.14,   8.67,   7.23, -23.29,  -7.06],
       [ -4.14,  53.86,  -8.67, -15.61, -10.59, -19.12],
       [  8.67,  -8.67,   0.  ,  -9.26, -37.47,   4.57],
       [  7.23, -15.61,  -9.26,  -1.14,   0.6 , -23.68],
       [-23.29, -10.59, -37.47,   0.6 ,   1.14,   4.  ],
       [ -7.06, -19.12,   4.57, -23.68,   4.  ,  -4.14]])

In [46]:
(2*500)**(1/3)

9.999999999999998

In [48]:
help(np.random.random)

Help on built-in function random:

random(...) method of numpy.random.mtrand.RandomState instance
    random(size=None)
    
    Return random floats in the half-open interval [0.0, 1.0). Alias for
    `random_sample` to ease forward-porting to the new random API.



In [8]:
help(Sphere)

Help on module agd.Sphere in agd:

NAME
    agd.Sphere

DESCRIPTION
    This module provides basic conversion utilities to manipulate low-dimensional spheres, 
    and related objects : rotations, quaternions, Pauli matrices, etc

FUNCTIONS
    ball3_from_rotation3(r, qRef=None)
        Produces an euclidean point from a rotation, 
        selecting in the intermediate step the quaternion 
        in the same hemisphere as qRef. (Defaults to southern.)
    
    pauli(a, b, c, d=None)
        Pauli matrix. Symmetric if d is None, Hermitian otherwise.
        Determinant is a^2-b^2-c^2-d^2
    
    plane_from_sphere(q)
        Produces a point in the equator plane from a point in the unit sphere.
    
    rotation3_from_ball3(e)
        Produces a rotation from an euclidean point. 
        Also returns the intermediate quaternion.
    
    rotation3_from_sphere3(q)
        Produces the rotation associated with a unit quaternion.
    
    sphere3_from_rotation3(r)
        Produces the uni